# Trained editors on exogenous-action world models — the full edit-type ablation

**Sevan, 2026-08-14.** Two world models trained on a world where **objects teleport during
training** — one that is *told* the teleport (actions as input) and one *observer* that is not —
plus a **control** trained with no actions and no teleports at all. Against all three, a thorough
ablation over every edit type this thread has: training-free structural editors, oracles, and
**trained** editors.

Two trained mechanisms, and the second is new:

* **Fine-tune** — the *world model* is adapted so that a **fixed, untrained** write works. Run twice:
  once with the Euclidean pseudoinverse, once with the **un-whitened (metric-corrected)** write from
  `../metric_corrected_edits/`. Same readout target, different metric, so this isolates the metric as
  a variable in the *trained* setting.
* **MLP editor `E_θ(h, start_pos, target_pos) → Δh`** — the *world model is frozen* and an editor
  network is trained. **This is not the published amortized editor**, which took `(h, target)` only:
  this one is also given where the objects *currently* are, so the displacement it must produce is
  supplied rather than inferred from `h`.

Each is trained under **two losses**, so rollout consistency is a measured variable:
**§2** the loss is next-step prediction RMSE at the edit frame (`k=1`); **§3** it is RMSE over the
next **8** free-run steps, forcing the edit to survive the dynamics.

> ### Sibling notebook
> **[`../action_hidden_size/action_hidden_size_sweep.ipynb`](../action_hidden_size/action_hidden_size_sweep.ipynb)**
> sweeps **hidden size** (8 → 512) across these same two exogenous families plus an endogenous one,
> with the **training-free** editors and oracles. It is where the capacity axis lives; **this**
> notebook is where the trained and fine-tuned editors live, at H=256.

**Registry:** `TRAINED_EDITOR_RUNS.md` · **Training:** `scripts/train_action_editors.py` ·
**Evaluation:** `scripts/eval_action_editors.py`

In [ ]:
# [1] setup
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

ROOT = Path("/home/sevan/research/physically-implicit-modeling")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/history_editing"))

from matplotlib.patches import Patch

from history_tools import ray_centroid, waterfall_grid   # the one waterfall spec, shared
from pim.figures.theme import style_ax

EVAL = ROOT / "runs/action_editors/eval"
MODELS = {
    "XG_A_H256": ("Exogenous teleport · actions given", "#0072B2"),
    "XG_C_H256": ("Exogenous teleport · observer (no action input)", "#56B4E9"),
    "CTRL_H256": ("Control · standard GRU · no actions, no teleports", "#5a5a5a"),
}
STANDARD = ["Unsteered", "Pseudoinverse Injection", "Metric-corrected Injection (un-whitened)",
            "Global PCA Projection", "Local PCA Geodesic", "MLP Grad Steering",
            "Multistep Steering @8"]
ORACLE = ["Oracle observation", "Counterfactual Overwriting", "Freeze-time TF @8",
          "Decoder Grad k=1", "Decoder Grad k=8", "Action interface"]
TRAINED_LBL = {
    "finetune": "Fine-tune · pseudoinverse write",
    "finetune_metric": "Fine-tune · un-whitened (metric-corrected) write",
    "mlp": "MLP editor  E(h, start, target)",
}
OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
      "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#5a5a5a"}
print("eval dir:", EVAL, "|", len(list(EVAL.glob("*.json"))), "model json files")

## Definitions

Every number is computed by `scripts/eval_action_editors.py`, which routes the §4 block through
`scripts/editability_metrics.py`. Nothing is re-derived here.

### The three world models (copied from `TRAINED_EDITOR_RUNS.md` — the notebook stands alone)

| code | label | training data | actions as input? |
|---|---|---|---|
| `XG_A_H256` | **Exogenous teleport · actions given · 256 hidden** | `datasets/7_cont_teleport`, 90k sequences, **objects teleport during training** (`p_action=0.30`, teleport to absolute coordinates), next-frame prediction, 400 epochs, seed 0 | **yes** |
| `XG_C_H256` | **Exogenous teleport · observer · 256 hidden** | identical data and recipe | **no** — must predict teleports it is never told about |
| `CTRL_H256` | **Control · standard GRU · 256 hidden** | `datasets/4_fixed_refl_inview` — **no actions, no teleports** | no |

### The trained arms — 18 = 3 models × 3 editors × 2 losses

All: 3000 steps, batch 64, Adam lr 1e-4, seed 0; editor training data disjoint from the reporting set.

| arm | what is TRAINED | what is FROZEN |
|---|---|---|
| **Fine-tune · pseudoinverse write** | the world model | the editor: linear-pseudoinverse readout injection through a probe fit once on the BASE model |
| **Fine-tune · un-whitened write** | the world model | the editor, but the fixed write is `Δ = Σ¹Wᵀ(WΣ¹Wᵀ + εI)⁻¹δ` — the metric-corrected form. Same readout target, different metric. |
| **MLP editor `E(h, start, target)`** | a 2×512 ReLU editor network | **the world model, entirely** |

`start` = the un-edited frame-`ef` positions (what the model would render if left alone);
`target` = the same world with the edited object teleported. Both flat `(x₀,y₀,x₁,y₁)`.

**Losses.** `k=1`: `MSE(decode(h_edited), gt_edited[ef])` — the edit **lands**.
`k=8`: `MSE(rollout(h_edited, 8), gt_edited[ef:ef+8])` — the edit **survives the dynamics**.

**Retention:** every fine-tune arm carries the prediction-retention term (next-step MSE on non-edit
sequences, weight 1.0). It separates "became editable" from "was destroyed and now echoes the
editor". The MLP arms need none — the world model is frozen, so their prediction is unchanged by
construction.

> ### ⚠ A fine-tuned arm is a DIFFERENT world model
> Its Edit Index cannot be read against the base model's unsteered row. Every fine-tune arm below
> therefore carries **its own unsteered row** and **its own next-step RMSE**, and the headline is the
> **gain over its own unsteered value**, reported beside the prediction cost. The MLP arms share the
> base model's unsteered row.

### Metrics

| name | formula | units | better | notes |
|---|---|---|---|---|
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)` over **differing** rays, per sample then averaged | −1…+1 | ↑ | +1 = the edited world, −1 = the unedited one, **≈0 = equidistant OR garbage**. Always read against that model's own unsteered row. |
| **gain over own unsteered** | `Edit Index(arm) − Edit Index(that model's Unsteered)` | index pts | ↑ | the only fair cross-model comparison when the world model itself differs. |
| **Target / Ghost / Collateral RMSE** | `RMSE(edited₀, gt_edited)` over that zone at step 0 | obs intensity | ↓ | appeared where it should / left where it was / left the other object alone. |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, gt_edited_s)` over K=15 | obs intensity | ↓ | achieved **and held** the true post-edit world. |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(its reference unsteered)` | ratio | ↓ | **> 1 = the edit left the rollout further from the truth than doing nothing.** |
| **next-step RMSE (vs clean)** | `RMSE(pred_{t+1}, clean_obs_{t+1})` | obs intensity | ↓ | the **prediction cost** of fine-tuning — the number that exposes an index bought by damage. |

### Editors in the ablation

**Standard / training-free:** Unsteered · Pseudoinverse Injection · **Metric-corrected Injection
(un-whitened)** · Global PCA Projection (POCS) · Local PCA Geodesic · MLP Grad Steering (frozen
1×128 probe, published defaults) · Multistep Steering @8 (the model's **own** decoded observations
fed back — never an external render).

**Oracle:** Oracle observation (one extra teacher-forced frame — **leads the others by one frame**,
labelled, never re-aligned) · Counterfactual Overwriting (8 rendered frames of a fabricated history
in which the object always travelled toward the target) · Freeze-time TF @8 · Decoder Grad k=1 ·
Decoder Grad k=8 · **Action interface** — `XG_A_H256` only, commanding the teleport through the
model's own action channel. The other two models have no action channel, so the bar is absent by
construction rather than missing.

In [ ]:
# [2] load every evaluation
R = {}
for m in MODELS:
    p = EVAL / f"{m}.json"
    if p.exists():
        R[m] = json.loads(p.read_text())
print(f"loaded {len(R)} models: {list(R)}")


def arm_key(model, kind, k):
    """kind in {finetune, finetune_metric, mlp}"""
    mid = "finetune__metric" if kind == "finetune_metric" else kind
    return f"{model}__{mid}__k{k}"


def trained(model, kind, k):
    return R[model]["trained"].get(arm_key(model, kind, k))


rows = ["| model | next-step RMSE (vs clean) | unsteered Edit Index | mean teleport |",
        "|---|---|---|---|"]
for m, (lbl, _) in MODELS.items():
    if m not in R:
        continue
    rows.append(f"| {lbl} | {R[m]['base_nextstep_rmse']:.4f} | "
                f"{R[m]['editors']['Unsteered']['edit_index']:+.3f} | "
                f"{R[m]['n_edits']} held-out edits |")
display(Markdown("**Table 1 — the three base models.** A better predictor's unsteered Edit Index "
                 "sits closer to −1, which is why every arm is read against its own row.\n\n"
                 + "\n".join(rows)))

## §1 — The baseline the trained editors have to beat

Nothing here is trained: the world models are exactly as they came out of next-frame-prediction
training. This section exists because **every trained arm in §2–§3 is read against these numbers** —
in particular against **metric-corrected injection**, which is the fixed write one of the fine-tuning
arms is trained to honour.

> **Overlap with the sibling notebook, stated plainly.** `../action_hidden_size/` also evaluates
> training-free editors on `XG_A_H256` and `XG_C_H256`, so the *Readout injection*, *Global PCA
> projection*, *MLP Grad Steering*, *Decoder Grad* and *Action interface* rows below are consistent
> with it rather than new. What is new here: **metric-corrected injection**, *Local PCA Geodesic*,
> *Multistep Steering*, *Oracle observation*, *Counterfactual Overwriting* and *Freeze-time TF* — the
> full oracle bracket the trained arms are measured inside — plus the dataset-4 control on the same
> editor set. Read §1 as this notebook's baseline, not as an independent finding.

In [ ]:
# [3] Fig 1 + Table 2 — the training-free and oracle ablation (the baseline the trained arms must beat)
# One canonical row order for every panel so the reader can scan HORIZONTALLY; a model that lacks an
# editor gets an explicit empty "n/a" slot rather than shifting every row beneath it.
ROW_ORDER = STANDARD + ORACLE

fig, axes = plt.subplots(1, len(R), figsize=(5.6 * len(R), 5.8), squeeze=False, sharey=True)
for j, (m, (lbl, col)) in enumerate([(m, MODELS[m]) for m in R]):
    ax = axes[0][j]
    style_ax(ax)
    ed = R[m]["editors"]
    y = np.arange(len(ROW_ORDER))
    for i, e in enumerate(ROW_ORDER):
        c = ed.get(e)
        if c is None:                       # keep the slot, say why it is empty
            ax.text(0.0, i, "  n/a — this model has no action channel", va="center", ha="left",
                    fontsize=7, color=OI["grey"], style="italic")
            continue
        colr = (OI["grey"] if e == "Unsteered" else (OI["green"] if e in ORACLE else OI["blue"]))
        degraded = c["fidelity_ratio"] > 1.05
        ax.barh(i, c["edit_index"], color=colr, height=0.7,
                hatch="///" if degraded else None,
                edgecolor=OI["red"] if degraded else "none", linewidth=1.0 if degraded else 0)
    ax.set_yticks(y)
    ax.tick_params(labelleft=(j == 0))
    ax.invert_yaxis()
    ax.axvline(0, color="k", lw=0.9)
    ax.axvline(ed["Unsteered"]["edit_index"], color=OI["grey"], lw=1.2, ls="--")
    ax.set_xlim(-0.95, 1.05)
    ax.set_xlabel("Edit Index — edit frame (step 0)")
    ax.set_title(lbl, fontsize=9.5)
handles = [Patch(facecolor=OI["blue"], label="training-free editor"),
           Patch(facecolor=OI["green"], label="oracle (needs ground truth)"),
           Patch(facecolor=OI["grey"], label="unsteered (and its dashed reference line)"),
           Patch(facecolor="white", hatch="///", edgecolor=OI["red"],
                 label="hatched = failed the fidelity guard (ratio > 1.05): the rollout ended "
                       "further from the truth than doing nothing")]
axes[0][0].set_yticklabels(ROW_ORDER, fontsize=8)
fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, 0.93))
fig.suptitle("Fig 1 — the baseline: training-free editors and oracles on the three world models "
             "(N=128 held-out edits)", fontsize=11, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.83])
fig.subplots_adjust(left=0.165)
plt.show()

hdr = "| editor | " + " | ".join(MODELS[m][0] for m in R) + " |"
rows = [hdr, "|" + "---|" * (len(R) + 1)]
for e in ROW_ORDER:
    cells = []
    for m in R:
        c = R[m]["editors"].get(e)
        cells.append("n/a" if c is None else
                     f"{c['edit_index']:+.3f} (fid {c['fidelity_ratio']:.2f})"
                     + (" ⚠" if c["fidelity_ratio"] > 1.05 else ""))
    rows.append(f"| {e} | " + " | ".join(cells) + " |")
display(Markdown("**Table 2 — training-free and oracle editors**, Edit Index at the edit frame "
                 "(step 0). ⚠ = fidelity ratio > 1.05, i.e. the edited rollout ended further from the "
                 "truth than doing nothing. `n/a` = that model has no action channel.\n\n"
                 + "\n".join(rows)))

## §2 — Trained editors, loss = **next-step prediction RMSE** (k = 1)

Both mechanisms trained so that the edit **lands on the very next frame**. Nothing constrains what
happens afterwards.

In [ ]:
# [4] Fig 2 + Table 3 — trained editors under the next-step loss
# ABSOLUTE Edit Index, same units as every other figure. A fine-tuned arm is a different world
# model, so its own unsteered value is marked on the SAME axis (grey caret) rather than being
# subtracted out — an editor can raise the index simply by degrading the frame toward "neither
# world", and only the absolute value plus that reference makes it visible.
def trained_panel(ax, k, title):
    style_ax(ax)
    kinds = ["finetune", "finetune_metric", "mlp"]
    width, x = 0.26, np.arange(len(R))
    for i, kind in enumerate(kinds):
        vals, refs, degraded = [], [], []
        for m in R:
            t = trained(m, kind, k)
            vals.append(np.nan if t is None else t["edit_index"])
            refs.append(np.nan if t is None else t["own_unsteered"])
            degraded.append(False if t is None else t["fidelity_ratio"] > 1.05)
        c = [OI["sky"], OI["purple"], OI["green"]][i]
        pos = x + (i - 1) * width
        ax.bar(pos, vals, width, color=c, label=TRAINED_LBL[kind],
               hatch=None, edgecolor="none")
        for xi, d in enumerate(degraded):
            if d:
                ax.bar(pos[xi], vals[xi], width, color="none", hatch="///",
                       edgecolor=OI["red"], linewidth=1.0)
        ax.scatter(pos, refs, marker="_", s=170, color=OI["grey"], zorder=4, linewidths=1.6)
    ax.set_xticks(x)
    ax.set_xticklabels([MODELS[m][0].replace(" · ", "\n") for m in R], fontsize=7.5)
    ax.set_ylabel("Edit Index — edit frame (step 0)")
    ax.set_title(title, fontsize=10)
    ax.axhline(0, color="k", lw=0.9)
    ax.set_ylim(-0.8, 0.45)


fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
trained_panel(axes[0], 1, "(a) trained with the next-step loss (k=1)")
trained_panel(axes[1], 8, "(b) trained with the 8-step rollout loss (k=8)")

ax = axes[2]
style_ax(ax)
x = np.arange(len(R))
ax.plot(x, [R[m]["base_nextstep_rmse"] for m in R], "o-", color=OI["grey"], lw=2, ms=8,
        label="base model (untouched)")
for kind, c, mk in [("finetune", OI["sky"], "s"), ("finetune_metric", OI["purple"], "D"),
                    ("mlp", OI["green"], "^")]:
    for k, ls in [(1, "-"), (8, "--")]:
        v = [trained(m, kind, k)["nextstep_rmse"] if trained(m, kind, k) else np.nan for m in R]
        ax.plot(x, v, marker=mk, ls=ls, color=c, lw=1.6, ms=6,
                label=f"{TRAINED_LBL[kind]}  k={k}")
ax.set_xticks(x)
ax.set_xticklabels([MODELS[m][0].replace(" · ", "\n") for m in R], fontsize=7.5)
ax.set_ylabel("next-step RMSE vs clean  (prediction cost)")
ax.set_title("(c) what the edit cost the world model", fontsize=10)
# handlelength so dashed and solid are actually distinguishable in the legend
ax.legend(fontsize=6.2, ncol=1, handlelength=3.4)

handles = [Patch(facecolor=OI["sky"], label=TRAINED_LBL["finetune"]),
           Patch(facecolor=OI["purple"], label=TRAINED_LBL["finetune_metric"]),
           Patch(facecolor=OI["green"], label=TRAINED_LBL["mlp"]),
           plt.Line2D([0], [0], marker="_", color=OI["grey"], lw=0, markersize=13,
                      markeredgewidth=1.6, label="that arm's own unsteered value"),
           Patch(facecolor="white", hatch="///", edgecolor=OI["red"],
                 label="hatched = failed the fidelity guard (> 1.05)")]
fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=7.5, frameon=False,
           bbox_to_anchor=(0.5, 0.93))
fig.suptitle("Fig 2 — trained editors: where each mechanism lands, and what it costs "
             "(N=128 held-out edits)", fontsize=11, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.82])
plt.show()


def trained_table(k, caption):
    rows = ["| model | mechanism | Edit Index (step 0) | its own unsteered | fidelity | "
            "Target RMSE | Ghost RMSE | next-step RMSE (base → arm) |",
            "|" + "---|" * 8]
    for m in R:
        for kind in ["finetune", "finetune_metric", "mlp"]:
            t = trained(m, kind, k)
            if t is None:
                continue
            cost = (f"{R[m]['base_nextstep_rmse']:.4f} → {t['nextstep_rmse']:.4f}"
                    if t["kind"] == "finetune" else
                    f"{R[m]['base_nextstep_rmse']:.4f} (unchanged — model frozen)")
            rows.append(f"| {MODELS[m][0]} | {TRAINED_LBL[kind]} | **{t['edit_index']:+.3f}** | "
                        f"{t['own_unsteered']:+.3f} | {t['fidelity_ratio']:.2f}"
                        f"{' ⚠' if t['fidelity_ratio'] > 1.05 else ''} | {t['target_rmse']:.3f} | "
                        f"{t['ghost_rmse']:.3f} | {cost} |")
    display(Markdown(caption + "\n\n" + "\n".join(rows)))


trained_table(1, "**Table 3 — trained editors, next-step loss (k=1).**")

## §3 — Trained editors, loss = **RMSE over the next 8 steps**

Identical mechanisms and budget; the loss now requires the edit to survive 8 free-run steps of the
model's own dynamics. Fig 2(b) already put this on the same axis as §2; the table gives the detail.

In [ ]:
# [5] Table 4 — trained editors under the rollout loss, and the k=1 vs k=8 trade
trained_table(8, "**Table 4 — trained editors, 8-step rollout loss (k=8).**")

rows = ["| model | mechanism | Edit Index k=1 → k=8 | fidelity k=1 → k=8 | GT-traj RMSE k=1 → k=8 |",
        "|---|---|---|---|---|"]
for m in R:
    for kind in ["finetune", "finetune_metric", "mlp"]:
        a, b_ = trained(m, kind, 1), trained(m, kind, 8)
        if a is None or b_ is None:
            continue
        rows.append(f"| {MODELS[m][0]} | {TRAINED_LBL[kind]} | "
                    f"{a['edit_index']:+.3f} → {b_['edit_index']:+.3f} | "
                    f"{a['fidelity_ratio']:.2f} → {b_['fidelity_ratio']:.2f} | "
                    f"{a['gt_traj_rmse']:.3f} → {b_['gt_traj_rmse']:.3f} |")
display(Markdown("**Table 5 — what the rollout loss trades.** The next-step loss optimises exactly "
                 "the frame the Edit Index scores; the rollout loss optimises the trajectory that "
                 "the fidelity ratio and GT-traj RMSE score. Read them together.\n\n"
                 + "\n".join(rows)))

## §4 — Edit Index across the rollout

The step-0 Edit Index says whether the edit **landed**; it says nothing about whether the model then
keeps the edited world. `../METRICS_AND_EDITORS.md` requires the by-step index wherever the step-0
index is reported, and this thread has a specific reason to want it: the `k=1` and `k=8` arms were
trained to optimise *different points on exactly this curve*, so the curve is where that design
choice becomes visible.

**Edit Index by step** is the same formula at every rollout step, scored against the counterfactual
world **rolled forward** (the edited object continuing along its own velocity, the other object on
its true trajectory) — so it stays bounded and comparable at every step, unlike an RMSE against a
static post-edit render.

One figure per world model, split by editor family so all ~19 methods stay legible. In the trained
panel, colour = mechanism and line style = loss (solid `k=1`, dashed `k=8`); the thin faint lines are
each **fine-tuned arm's own unsteered curve**, since those arms are different world models and cannot
be read against the base model's row.

In [ ]:
# [6] Fig 3 — Edit Index across the K=15 rollout, one figure per world model
STD_COL = {"Unsteered": OI["grey"], "Pseudoinverse Injection": OI["blue"],
           "Metric-corrected Injection (un-whitened)": OI["purple"],
           "Global PCA Projection": OI["sky"], "Local PCA Geodesic": OI["yellow"],
           "MLP Grad Steering": OI["orange"], "Multistep Steering @8": OI["red"]}
# NOTE: Decoder Grad k=1 and k=8 must be visually separable — Okabe-Ito orange (#E69F00) and
# vermillion (#D55E00) are too close at line width, so k=8 gets black.
ORA_COL = {"Oracle observation": OI["sky"], "Counterfactual Overwriting": OI["green"],
           "Freeze-time TF @8": OI["blue"], "Decoder Grad k=1": OI["orange"],
           "Decoder Grad k=8": "#000000", "Action interface": OI["purple"]}
TR_COL = {"finetune": OI["sky"], "finetune_metric": OI["purple"], "mlp": OI["green"]}

for m in R:
    ed = R[m]["editors"]
    steps = np.arange(len(ed["Unsteered"]["edit_index_by_step"]))
    uns = np.array(ed["Unsteered"]["edit_index_by_step"])
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.9), sharey=True)
    for ax in axes:
        style_ax(ax)
        ax.axhline(0, color="k", lw=0.9)
        ax.plot(steps, uns, color=OI["grey"], lw=2.2, ls="--", label="Unsteered (base model)")
        ax.set_xlabel("rollout step after the edit  (step 0 = the edit frame)")
        ax.set_ylim(-1.02, 1.02)

    for e, c in STD_COL.items():
        if e in ed and e != "Unsteered":
            axes[0].plot(steps, ed[e]["edit_index_by_step"], color=c, lw=1.8, marker="o", ms=3,
                         label=e)
    axes[0].set_ylabel("Edit Index  (+1 = edited world, −1 = unedited)")
    axes[0].set_title("(a) training-free", fontsize=10)
    axes[0].legend(fontsize=6.4, loc="upper center", bbox_to_anchor=(0.5, -0.14),
               ncol=2, frameon=False)

    for e, c in ORA_COL.items():
        if e in ed:
            axes[1].plot(steps, ed[e]["edit_index_by_step"], color=c, lw=1.8, marker="s", ms=3,
                         label=e)
    axes[1].set_title("(b) oracle", fontsize=10)
    axes[1].legend(fontsize=6.4, loc="upper center", bbox_to_anchor=(0.5, -0.14),
               ncol=2, frameon=False)

    shown_own = False
    for kind, c in TR_COL.items():
        for k, ls in [(1, "-"), (8, "--")]:
            t = trained(m, kind, k)
            if t is None:
                continue
            short = {"finetune": "Fine-tune · pseudoinverse",
                     "finetune_metric": "Fine-tune · un-whitened",
                     "mlp": "MLP editor E(h,start,target)"}[kind]
            axes[2].plot(steps, t["edit_index_by_step"], color=c, lw=2.0, ls=ls,
                         label=f"{short}  k={k}")
            if kind != "mlp":       # fine-tunes are different models — show their own baseline
                axes[2].plot(steps, t["own_unsteered_by_step"], color=c, lw=0.9, ls=":", alpha=0.5,
                             label="its own unsteered" if not shown_own else None)
                shown_own = True
    axes[2].set_title("(c) trained  (solid k=1, dashed k=8)", fontsize=10)
    axes[2].legend(fontsize=6.4, loc="upper center", bbox_to_anchor=(0.5, -0.14),
               ncol=2, frameon=False)

    fig.suptitle(f"Fig 3 — {MODELS[m][0]}: Edit Index across the {len(steps)}-step rollout "
                 f"(N={R[m]['n_edits']} held-out edits)", fontsize=11)
    fig.tight_layout(rect=[0, 0.02, 1, 0.93])
    out = Path("/tmp/trained_editors_actions")
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{m}_index_by_step.png", dpi=120, bbox_inches="tight")
    plt.show()

rows = ["| model | mechanism | step 0 | step 4 | step 8 | step 14 | **peak (at step)** | "
        "gain vs own unsteered @ step 14 |", "|" + "---|" * 8]
for m in R:
    base_uns = np.array(R[m]["editors"]["Unsteered"]["edit_index_by_step"])
    entries = [("Unsteered", R[m]["editors"]["Unsteered"]["edit_index_by_step"], base_uns),
               ("Counterfactual Overwriting (oracle)",
                R[m]["editors"]["Counterfactual Overwriting"]["edit_index_by_step"], base_uns)]
    for kd in TR_COL:
        for k in (1, 8):
            t = trained(m, kd, k)
            if t:
                entries.append((f"{TRAINED_LBL[kd]} k={k}", t["edit_index_by_step"],
                                np.array(t["own_unsteered_by_step"])))
    for label, series_, ref in entries:
        v = np.array(series_)
        pk = int(v.argmax())
        rows.append(f"| {MODELS[m][0]} | {label} | {v[0]:+.3f} | {v[4]:+.3f} | {v[8]:+.3f} | "
                    f"{v[-1]:+.3f} | **{v[pk]:+.3f}** (step {pk}) | {v[-1] - ref[-1]:+.3f} |")
display(Markdown(
    "**Table 6 — Edit Index along the rollout.** Note the **Unsteered** row itself climbs "
    "(e.g. −0.671 → −0.438 on the control): a free-running model drifts away from both reference "
    "worlds, so every curve rises somewhat for reasons that have nothing to do with the edit. That "
    "is why the last column gives the **gap to that arm's own unsteered curve** at step 14 — for a "
    "fine-tuned arm the reference is its own model's unsteered curve, not the base model's. "
    "A step-14 ÷ step-0 'retention' ratio is deliberately **not** reported here: several arms have a "
    "step-0 index near zero, where that ratio explodes or flips sign and means nothing.\n\n"
    + "\n".join(rows)))

## §5 — Observation space

Required by `CLAUDE.md` for any claim about an effect on the generations, through the same
`waterfall_grid(...)` helper as the other threads on this branch. Gray on dark; a GT column of clean
sim observations; six **noisy** pre-edit context frames above the dashed edit line; below it, each
column is its own free-run from step 0. Solid green = target, dashed red = ghost.

Shown for the **action-conditioned** model and the **control**, with the best editor of each family
side by side.

In [ ]:
# [7] Fig 4 — waterfalls
def draw(m, title):
    z = np.load(EVAL / f"{m}_rollouts.npz")
    rolls = {k.split("::", 1)[1]: z[k] for k in z.files if k.startswith("roll::")}
    # every mechanism under discussion gets a column — including both fine-tune writes
    want = ["Unsteered", "Pseudoinverse Injection", "Metric-corrected Injection (un-whitened)",
            arm_key(m, "finetune", 1), arm_key(m, "finetune_metric", 1),
            arm_key(m, "mlp", 1), arm_key(m, "mlp", 8),
            "Action interface", "Counterfactual Overwriting"]
    order = [w for w in want if w in rolls]
    short = {arm_key(m, "finetune", 1): "Fine-tune · pseudoinverse\nwrite (k=1)",
             arm_key(m, "finetune_metric", 1): "Fine-tune · un-whitened\nwrite (k=1)",
             arm_key(m, "mlp", 1): "MLP editor\nE(h,start,target) k=1",
             arm_key(m, "mlp", 8): "MLP editor\nE(h,start,target) k=8"}
    lab = {}
    for w in order:
        t = R[m]["trained"].get(w) or R[m]["editors"].get(w)
        nm = short.get(w, w)
        lab[w] = f"{nm}\nEdit Index {t['edit_index']:+.2f} · fid {t['fidelity_ratio']:.2f}"
    samples = list(np.argsort(z["teleport"])[::-1][:3])   # >= 3 rows, always
    fig = waterfall_grid(rolls={w: rolls[w] for w in order}, ctx=z["ctx"], gt_roll=z["gt_roll"],
                         tgt_cx=ray_centroid(z["tgt_mask"]), ghost_cx=ray_centroid(z["ghost_mask"]),
                         samples=samples, edit_frame=20,
                         leads_by_one=("Oracle observation",), title=title, labels=lab)
    out = Path("/tmp/trained_editors_actions")
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{m}_waterfall.png", dpi=120, bbox_inches="tight", facecolor="#0a0a14")
    plt.show()


for m in ["XG_A_H256", "XG_C_H256", "CTRL_H256"]:
    if m in R:
        draw(m, f"Fig 4 — {MODELS[m][0]} — each column is that arm's own free-run\n"
                f"(two largest teleports; N=128 held-out edits)")

## Summary

**What this notebook measures** (invariant): every edit type this thread has — training-free,
oracle, and **trained** — against three world models that differ in whether teleports and actions
were present during training, under two editor-training losses.

### Current results (updated 2026-08-14)

**1. The new MLP editor is the first probe-free latent editor in the thread to cross zero, and it
costs the world model nothing.** With the world model **completely frozen**, `E_θ(h, start, target)`
reaches Edit Index **+0.204 / +0.111 / +0.117** (control / actions-given / observer) against
unsteered rows of −0.671 / −0.669 / −0.578 — gains of **+0.875 / +0.780 / +0.695** at fidelity
0.84 / 0.89 / 0.82, with Target RMSE roughly halved (0.48 → 0.25–0.28). For comparison, the
previously published best learned mechanism was the amortized `E(h, target)` at **−0.14**, and the
best training-free structural editor in this notebook is metric-corrected injection at −0.42…−0.52.
Fig 3 shows it is a real relocation in observation space — the object appears on the green target
locator and the ghost dims — though with visible streaking the counterfactual oracle does not have,
which is what the 0.84 fidelity is measuring.

*The difference from the published editor is the extra input.* Giving the network the **starting**
positions means it is handed the displacement rather than having to infer the current world from
`h`. That is the only change, and it moves the mechanism from −0.14 to +0.20.

**2. The un-whitened (metric-corrected) write beats the plain pseudoinverse as a fine-tuning target —
consistently, on all three models and both losses.** Gains over each arm's own unsteered row:

| loss | model | pseudoinverse write | **un-whitened write** |
|---|---|---|---|
| k=1 | control | +0.233 | **+0.341** |
| k=1 | actions given | +0.187 | **+0.292** |
| k=1 | observer | +0.182 | **+0.302** |
| k=8 | control | +0.141 | **+0.263** |
| k=8 | actions given | +0.126 | **+0.178** |
| k=8 | observer | +0.109 | **+0.241** |

and it is marginally *cheaper* in prediction too (control 0.1218 vs 0.1253). So the metric correction
— derived as undoing the `Σ_hh⁻¹` whitening baked into a least-squares probe — is not only the best
training-free structural editor (reproduced here at −0.516 / −0.516 / −0.423 vs pseudoinverse
−0.656 / −0.649 / −0.552), it is also a **better thing to fine-tune a model towards**. Six of six
cells agree.

**3. Fine-tuning always costs prediction; the MLP editor never does.** Every fine-tune arm degrades
next-step RMSE — control 0.1041 → 0.1218–0.1253 (+17–20%), actions-given 0.1071 → 0.1205–0.1356
(+13–27%) — and still ends **negative** on the Edit Index. The MLP arms leave the world model
untouched, so their prediction is unchanged by construction, and they end positive. On this
evidence, adapting the *editor* is strictly better than adapting the *model*.

**4. The two losses do NOT simply trade index for fidelity — the rollout-trained arms *overtake*.**
At step 0 the picture looks like a trade (control: MLP editor **+0.204** at `k=1` vs **−0.035** at
`k=8`, fidelity 0.84 vs 0.67). The by-step curves in Fig 3 show that reading is incomplete: the
`k=8` arms start lower and then **cross above their `k=1` counterparts by about step 4 and stay
there for the rest of the rollout**, in every mechanism and every model.

| control · mechanism | step 0 | step 4 | step 8 | step 14 |
|---|---|---|---|---|
| MLP editor `k=1` | **+0.204** | +0.215 | +0.223 | +0.127 |
| MLP editor `k=8` | −0.035 | **+0.267** | **+0.286** | **+0.230** |
| Fine-tune · un-whitened `k=1` | −0.224 | −0.196 | −0.160 | −0.100 |
| Fine-tune · un-whitened `k=8` | −0.339 | −0.041 | **+0.041** | **+0.042** |

So the rollout loss does not merely buy trajectory fidelity at the cost of landing: it produces an
edit that **takes a few steps to materialise and then holds**, ending both higher on the index *and*
lower on GT-traj RMSE. The `k=1` arms do the opposite — they land hardest on exactly the frame they
were trained on and then decay. **Only the step-0 number makes `k=1` look better;** on any horizon
past ~4 steps `k=8` is the better editor. Reporting the step-0 index alone would have inverted this
conclusion, which is precisely why the registry requires the by-step curve alongside it.

*Caveat the curves also make visible:* the **unsteered** row climbs on its own (control −0.671 →
−0.438) because a free-running model drifts away from both reference worlds. Part of every arm's
rise is that drift, which is why Table 6's last column reports the **gap to that arm's own unsteered
curve** rather than the raw value.

**5. Actions and teleports in the world model's training buy nothing for latent editing.** The
control — trained with **no actions and no teleports at all** — is not worse at any of this; it is
the *best* cell for both the MLP editor (+0.875) and the metric fine-tune (+0.341). The
action-conditioned model's advantage shows up only in its **action interface** (+0.618, fidelity
0.71), the channel that bypasses the latent entirely.

### Interpretation (mine, not established)

The thread's negative has always been about **probe-derived** writes: a direction that correlates
with position, inverted. Everything in §1 is that, and everything in §1 fails. The MLP editor is the
first mechanism to be handed the *edit* (start → target) and allowed to learn its own write into a
**frozen** state, and it is the first to produce a positive Edit Index. That does not overturn the
negative — it locates it. What was missing was never reachability or capacity (see
`../full_rowspace_edit/`, `../action_hidden_size/`) but a **map from the intended change to the
state change**, which a linear probe's pseudoinverse is a poor estimate of and which a small network
can learn from examples.

That reading also explains result 2: the un-whitened write is a better *approximation* of that map
than the Euclidean pseudoinverse, so a model fine-tuned toward it has less to correct.

The honest limit: +0.20 is "just past equidistant", not the oracle's +0.63, and the k=8 arm shows
that holding the edited world costs the landing. The mechanism edits; it does not yet edit cleanly.

### Owed / scope limits

* One seed per cell; 3000 steps for every arm; a single editor architecture (2×512) and one width.
* The MLP editor is given **ground-truth** start and target positions. A deployable version would
  read the start from the model's own probe — untested, and the gap between the two is the obvious
  next measurement.
* Held-out by construction (disjoint edit pools), but **within one edit distribution**: no test of
  generalisation to a withheld object or to displacements outside the training range, which is where
  the published fine-tuning arms failed ("it is a button, not a handle").
* `Oracle observation` varies sharply across models (+0.013 control, +0.234 actions-given, **+0.557**
  observer) — the observer twin gains most from simply being shown the post-edit frame. Unexplained.
* Fine-tuning used retention weight 1.0 throughout; the weight was not swept.